In [3]:
import pandas as pd
import numpy as np
import os
from dotenv import load_dotenv

load_dotenv()

True

In [4]:
BUCKET_NAME = 'akshat-funnel-project-bucket' 
FILE_KEY = 'raw/ecommerce_funnel_data.csv'

df = pd.read_csv(
    f's3://{BUCKET_NAME}/{FILE_KEY}',
    storage_options={
        "key": os.getenv("ACCESS_KEY"),
        "secret": os.getenv("SECRET_KEY")
    }
)

In [5]:
df.head(10)

,session_id,user_id,timestamp,traffic_source,device_type,country,product_category,visited_site,viewed_product,added_to_cart,initiated_checkout,completed_purchase,page_views,session_duration_sec,cart_value,revenue
0,SES_000001,USER_1177,2025-12-31T11:47:30.498Z,Facebook,desktop,Australia,Home & Garden,Yes,Yes,No,No,No,8,204.0,0.00,0.0
1,SES_000002,USER_900,2025-12-27T02:22:57.772Z,Email,Tablet,NaN,NaN,Yes,No,No,No,No,2,813.0,0.00,NaN
2,SES_000003,USER_1481,2025-12-26T00:23:08.107Z,email,desktop,NaN,NaN,Yes,No,No,No,No,3,867.0,0.00,0.0
3,SES_000004,USER_4,2025-12-17T09:05:35.276Z,Google,desktop,Australia,Fashion,Yes,Yes,Yes,No,No,12,563.0,250.59,0.0
4,SES_000005,USER_5,2025-12-12T18:52:57.668Z,Organic Search,mobile,Australia,Beauty,Yes,Yes,No,No,No,3,778.0,0.00,0.0
5,SES_000006,USER_6,2025-12-23T05:23:54.105Z,Email,Tablet,France,Books,Yes,Yes,Yes,No,No,13,1412.0,495.08,0.0
6,SES_000007,USER_7,2025-12-23T07:37:21.781Z,Google,mobile,UK,NaN,Yes,No,No,No,No,1,1630.0,0.00,0.0
7,SES_000008,USER_32,2025-12-13T08:31:35.179Z,email,desktop,India,NaN,Yes,No,No,No,No,3,514.0,0.00,0.0
8,SES_000009,USER_9,2025-12-15T06:18:02.597Z,Facebook,Mobile,India,Beauty,Yes,Yes,No,No,No,7,1236.0,0.00,0.0
9,SES_000010,USER_10,2025-12-06T21:44:57.959Z,google,desktop,Canada,Home & Garden,Yes,Yes,Yes,No,No,14,853.0,95.27,0.0


In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 16 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   session_id            5000 non-null   object 
 1   user_id               5000 non-null   object 
 2   timestamp             4903 non-null   object 
 3   traffic_source        5000 non-null   object 
 4   device_type           4228 non-null   object 
 5   country               4519 non-null   object 
 6   product_category      2882 non-null   object 
 7   visited_site          5000 non-null   object 
 8   viewed_product        5000 non-null   object 
 9   added_to_cart         5000 non-null   object 
 10  initiated_checkout    5000 non-null   object 
 11  completed_purchase    5000 non-null   object 
 12  page_views            5000 non-null   int64  
 13  session_duration_sec  4845 non-null   float64
 14  cart_value            4967 non-null   float64
 15  revenue              

In [7]:
df.describe()

,page_views,session_duration_sec,cart_value,revenue
count,5000.000000,4845.000000,4967.000000,4930.000000
mean,5.876600,1093.042724,70.205160,28.530673
std,4.510814,1082.422417,141.252785,93.051199
min,1.000000,0.000000,0.000000,0.000000
25%,2.000000,503.000000,0.000000,0.000000
50%,4.000000,975.000000,0.000000,0.000000
75%,10.000000,1422.000000,38.680000,0.000000
max,15.000000,9989.000000,529.860000,497.380000


In [8]:
df2 = df.copy()

In [9]:
# Handling missing timestamps
df['timestamp'] = df['timestamp'].replace('', np.nan)
median_date = '2024-12-16T12:00:00.000Z'
df['timestamp'] = df['timestamp'].fillna(median_date)

In [10]:
df['timestamp'].notna().sum()

np.int64(5000)

In [11]:
# Standardize traffic_source
source_mapping = {
    'google': 'Google',
    'facebook': 'Facebook',
    'instagram': 'Instagram',
    'email': 'Email',
    'direct': 'Direct',
    'organic search': 'Organic Search',
    'referral': 'Referral'
}
df['traffic_source'] = df['traffic_source'].str.lower().map(source_mapping).fillna(df['traffic_source'])

In [12]:
df['traffic_source'].unique()

array(['Facebook', 'Email', 'Google', 'Organic Search', 'Direct',
       'Instagram', 'Referral'], dtype=object)

In [13]:
# Standardize device_type
device_mapping = {
    'mobile': 'Mobile',
    'desktop': 'Desktop',
    'tablet': 'Tablet'
}
df['device_type'] = df['device_type'].replace('', np.nan)
df['device_type'] = df['device_type'].str.lower().map(device_mapping)
df['device_type'] = df['device_type'].replace(np.nan, 'Unknown')

In [14]:
df['device_type'].unique()

array(['Desktop', 'Tablet', 'Mobile', 'Unknown'], dtype=object)

In [15]:
df['country'].unique()

array(['Australia', nan, 'France', 'UK', 'India', 'Canada', 'Germany',
       'US', 'USA', 'United States'], dtype=object)

In [16]:
# Standardize country
country_mapping = {
    'us': 'USA',
    'united states': 'USA',
    'usa': 'USA',
    'uk': 'UK',
    'canada': 'Canada',
    'germany': 'Germany',
    'france': 'France',
    'australia': 'Australia',
    'india': 'India'
}
df['country'] = df['country'].replace('', np.nan)
df['country'] = df['country'].str.lower().map(country_mapping)
df['country'] = df['country'].replace(np.nan, 'Unknown')

In [17]:
df['country'].unique()

array(['Australia', 'Unknown', 'France', 'UK', 'India', 'Canada',
       'Germany', 'USA'], dtype=object)

In [18]:
# Handle product_category
df.loc[(df['viewed_product'] == 'Yes') & (df['product_category'].isna() | (df['product_category'] == '')), 
       'product_category'] = 'Uncategorized'

In [19]:
df.loc[(df['viewed_product'] == 'No') & ((df['product_category'] == '') | ((df['product_category'].isna() ))), 'product_category'] = 'N/A'

In [20]:
mapping_dict = {'Yes': 1, 'No': 0}

columns_to_map = ['visited_site', 'viewed_product', 'added_to_cart', 'initiated_checkout', 'completed_purchase']

df[columns_to_map] = df[columns_to_map].replace(mapping_dict)

C:\Users\aksha\AppData\Local\Temp\ipykernel_19940\896851326.py:5: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df[columns_to_map] = df[columns_to_map].replace(mapping_dict)


In [21]:
Q1 = df['session_duration_sec'].quantile(0.25)
Q3 = df['session_duration_sec'].quantile(0.75)
IQR = Q3 - Q1
upper_bound = Q3 + 1.5 * IQR
print('Upper bound : ', upper_bound)
outliers = (df['session_duration_sec'] > upper_bound).sum()
print('Total Outliers : ', outliers)

Upper bound :  2800.5
Total Outliers :  140


In [22]:
df.loc[df['session_duration_sec'] >= upper_bound, 'session_duration_sec'] = upper_bound

In [23]:
median_value = df['session_duration_sec'].median(skipna=True)
df['session_duration_sec'] = df['session_duration_sec'].fillna(median_value)

In [24]:
# Cleaning revenue data & cart value
df['revenue'] = df['revenue'].replace(['N/A', '', 'null'], 0)
df['revenue'] = pd.to_numeric(df['revenue'], errors='coerce').fillna(0)

df['cart_value'] = df['cart_value'].replace(['N/A', '', 'null'], 0)
df['cart_value'] = pd.to_numeric(df['cart_value'], errors='coerce').fillna(0)

In [25]:
# Ensuring funnel consistency 
consistency_fixes = 0
for idx, row in df[df['completed_purchase'] == 'Yes'].iterrows():
    if row['viewed_product'] != 'Yes':
        df.at[idx, 'viewed_product'] = 'Yes'
        consistency_fixes += 1
    if row['added_to_cart'] != 'Yes':
        df.at[idx, 'added_to_cart'] = 'Yes'
        consistency_fixes += 1
    if row['initiated_checkout'] != 'Yes':
        df.at[idx, 'initiated_checkout'] = 'Yes'
        consistency_fixes += 1
print('Fixed rows : ', consistency_fixes)

Fixed rows :  0


In [26]:
df['page_views'] = pd.to_numeric(df['page_views'], errors='coerce')

In [27]:
mapping_dict = {'Yes': 1, 'No': 0}

columns_to_map = ['visited_site', 'viewed_product', 'added_to_cart', 'initiated_checkout', 'completed_purchase']

df[columns_to_map] = df[columns_to_map].replace(mapping_dict)

# Feature Engineering

In [28]:
# Extract datetime features
df['timestamp_parsed'] = pd.to_datetime(df['timestamp'], errors='coerce')
df['hour_of_day'] = df['timestamp_parsed'].dt.hour
df['day_of_week'] = df['timestamp_parsed'].dt.dayofweek  # 0=Monday
df['day_name'] = df['timestamp_parsed'].dt.day_name()
df['is_weekend'] = np.where(df['day_of_week'].isin([5, 6]), 1, 0)

In [29]:
df.columns

Index(['session_id', 'user_id', 'timestamp', 'traffic_source', 'device_type',
       'country', 'product_category', 'visited_site', 'viewed_product',
       'added_to_cart', 'initiated_checkout', 'completed_purchase',
       'page_views', 'session_duration_sec', 'cart_value', 'revenue',
       'timestamp_parsed', 'hour_of_day', 'day_of_week', 'day_name',
       'is_weekend'],
      dtype='object')

In [30]:
# Bounce rate indicator
df['is_bounce'] = ((df['page_views'] <= 1) & (df['session_duration_sec'] <= 30)).astype(int)

# Cart abandonment flag
df['cart_abandoned'] = ((df['added_to_cart'] == 1) & (df['completed_purchase'] != 1)).astype(int)

# Checkout abandonment flag
df['checkout_abandoned'] = ((df['initiated_checkout'] == 1) & (df['completed_purchase'] != 1)).astype(int)

In [31]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 24 columns):
 #   Column                Non-Null Count  Dtype              
---  ------                --------------  -----              
 0   session_id            5000 non-null   object             
 1   user_id               5000 non-null   object             
 2   timestamp             5000 non-null   object             
 3   traffic_source        5000 non-null   object             
 4   device_type           5000 non-null   object             
 5   country               5000 non-null   object             
 6   product_category      5000 non-null   object             
 7   visited_site          5000 non-null   int64              
 8   viewed_product        5000 non-null   int64              
 9   added_to_cart         5000 non-null   int64              
 10  initiated_checkout    5000 non-null   int64              
 11  completed_purchase    5000 non-null   int64              
 12  page_v

In [34]:
import csv
df.to_csv(
    './processed/cleaned_funnel_data.csv', 
    index=False,            # Saves time by not writing a column for row numbers
    quoting=csv.QUOTE_MINIMAL, # Only puts quotes around cells that actually need them
    chunksize=1000,         # Writes in batches to prevent memory spikes
    encoding='utf-8',       # Standard encoding for "heavy" text
    escapechar='\\'         # Helps handle special characters in heavy columns faster
)

# Importing data to mysql

In [ ]:
from sqlalchemy import create_engine, text

USER = "root"
PASSWORD = os.getenv("DB_PASSWORD")
HOST = "localhost"
PORT = "3306"
DATABASE = "ecommerce_funnel_db"

server_url = f"mysql+pymysql://{USER}:{PASSWORD}@{HOST}:{PORT}"
server_engine = create_engine(server_url)

with server_engine.connect() as conn:
    conn.execute(text(f"CREATE DATABASE IF NOT EXISTS {DATABASE}"))
    print(f"Database '{DATABASE}' created successfully.")

db_url = f"mysql+pymysql://{USER}:{PASSWORD}@{HOST}:{PORT}/{DATABASE}"
db_engine = create_engine(db_url)

try:
    df.to_sql(
        name='cleaned_funnel_data', 
        con=db_engine, 
        if_exists='replace',    
        index=False,         
        chunksize=1000       
    )
    print(f"Success! {len(df)} rows imported to MySQL table: 'cleaned_funnel_data'")
except Exception as e:
    print(f"Error during data import: {e}")

Database 'ecommerce_funnel_db' created successfully.
Success! 5000 rows imported to MySQL table: 'cleaned_funnel_data'
